# Day 24 — Lecture 2: Dummy Variable Trap

> Source: Apna College Prime AI/ML • Day 24 Lecture 2 • Video Duration: `06:09`
> Core Objective: Multicollinearity (Dummy Variable Trap) samajhna aur `drop_first=True` use karke redundant dummy column remove karna.
> Key Parameters: `columns=["region"]`, `drop_first=True`, `dtype=int`
> Dataset Used: `insurance.csv`
> Core Concept: Agar $k$ categories hain toh sirf $k - 1$ dummy columns kaafi hain — missing $k$-th category baaki sab ke `0` hone se implicit ho jati hai

## 🛠️ Step 0: Setup & Imports

Chalo libraries import karke `insurance.csv` load karte hain aur `region` ki unique categories check karte hain.

In [ ]:
# 1. Libraries import karo
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 2. Dataset load karo
insurance_data = pd.read_csv("insurance.csv")

# 3. Region ki 4 unique categories verify karo
print("Unique regions:", insurance_data["region"].unique())

## ⚠️ Step 1: Multicollinearity & Dummy Variable Trap

Lecture me samjhaya gaya: Jab ek column doosre columns se exact math equation se nikal sake, toh **Multicollinearity** paida hoti hai.
- **Equation:** `northeast + northwest + southeast + southwest = 1`
- **Problem:** Linear model me $h_\theta(x) = \theta_0 + \theta_1 x_1 + \dots$ ke weights ($\theta$) unstable aur galat ho sakte hain kyunki features aapas me highly correlated ho jate hain.

In [ ]:
# 1. drop_first=False ke saath 4 dummy columns banao
X_full = pd.get_dummies(
    insurance_data.drop(columns=["charges"]),
    columns=["region"],
    drop_first=False,
    dtype=int
)

# 2. Prove karo ki region_northeast baaki 3 columns se predict ho sakta hai
reconstructed = 1 - (X_full["region_northwest"] + X_full["region_southeast"] + X_full["region_southwest"])
matches = (X_full["region_northeast"] == reconstructed).all()

# 3. Exact dependency confirm karo
print("100% Exact Redundancy exists:", matches)

## 📍 Step 2: Solution — `drop_first=True` Apply Karo

Dekho trap se bachne ke liye hum ek redundant column drop kar dete hain (`drop_first=True`).

| `region_northwest` | `region_southeast` | `region_southwest` | Implicit Region |
|:---:|:---:|:---:|---|
| `0` | `0` | `0` | `northeast` (Dropped column) |
| `1` | `0` | `0` | `northwest` |
| `0` | `1` | `0` | `southeast` |

In [ ]:
# 1. Features aur target alag karo
X = insurance_data.drop(columns=["charges"])
y = insurance_data["charges"]

# 2. drop_first=True lagakar pehla dummy column (northeast) drop karo
X = pd.get_dummies(X, columns=["region"], drop_first=True, dtype=int)

# 3. Sex aur smoker map karo
X["sex"] = X["sex"].map({"female": 1, "male": 0})
X["smoker"] = X["smoker"].map({"yes": 1, "no": 0})
print("Remaining columns:", list(X.columns))

## 🔍 Step 3: Implicit Category Verify Karo (`X.sample(10)`)

Chalo random rows sample karke dekhte hain jahan teeno bache hue region columns `0` hon — wahan user automatically `northeast` se belong karta hai.

In [ ]:
# 1. 10 random rows sample karke region columns inspect karo
print(X.sample(10, random_state=42)[["region_northwest", "region_southeast", "region_southwest"]])

# 2. Aisi rows filter karo jahan teeno remaining region columns 0 hain
implicit_ne = X[(X["region_northwest"] == 0) & (X["region_southeast"] == 0) & (X["region_southwest"] == 0)]
print("Implicit northeast rows count:", len(implicit_ne))

## 📈 Step 4: Model Retrain After Trap Fix

Ab clean features (bina redundancy ke) par dobara `LinearRegression()` train karke `r2_score` check karte hain — accuracy bilkul same rehti hai aur model mathematically stable rehta hai.

In [ ]:
# 1. Train-test split karo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Model train karke predictions lo
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# 3. R-squared score verify karo
r2 = r2_score(y_test, y_pred)
print("r2 score:", r2)

## 🏆 Step 5: Practice Problem — 7 Days of the Week

Lecture ke end ka pause-and-think sawal: Agar dataset me `day` column ke andar hafte ke 7 din (`Mon`–`Sun`) hon, toh One-Hot Encoding + Dummy Variable Trap fix ke baad kitne columns banenge?
- **Answer:** $k - 1 = 7 - 1 = 6$ dummy columns.

In [ ]:
# 1. 7 days of week ka sample dataframe banao
days_df = pd.DataFrame({
    "day": ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
})

# 2. drop_first=True se One-Hot Encoding apply karo
encoded_days = pd.get_dummies(days_df, columns=["day"], drop_first=True, dtype=int)

# 3. Final dummy columns count aur names verify karo
print("Total dummy columns for 7 days:", encoded_days.shape[1])
print(list(encoded_days.columns))